In [ ]:
%load_ext autoreload
%autoreload 2

# Further Analysis

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
from Code.Utils.util_methods import UtilMethods
from keras.models import load_model
from Code.NN.utils.keras_functions import binary_focal_loss
import json
from matplotlib import pyplot as plt



base = UtilMethods.find_project_root(os.getcwd())
print(f"Project root found: {base}")

if load_dotenv(f'{base}/.env'):
    print(".env found")
else:
    print("ERROR .env not found")

In [ ]:
MODELS_PATH = f'{base}/Code/NN/Results/Tuning/prc-split-val-small-threshold-04'
MODEL_NAME = 'tuned_model_60'
RE_EVALUATE_ALL = False # re evaluates all the models (keep false)
THRESHOLD = 0.4 # DO NOT MODIFY

In [ ]:
# Re-evaluate all models if needed

In [ ]:
from Code.NN.NN_tuner import re_evaluate_models


if RE_EVALUATE_ALL:
    re_evaluate_models(MODELS_PATH, threshold=0.4, predict=False)

## Load the test data

In [ ]:
X_test = pd.read_csv(f'{base}/Dataset/traintest/X_test.csv')
y_test = pd.read_csv(f'{base}/Dataset/traintest/y_test.csv')

In [ ]:
print(f'Size of the test dataset: {len(X_test)}')

## Load the model to use and the history

In [ ]:
model_folder_path = f"{MODELS_PATH}/{MODEL_NAME}"

model = load_model(f"{model_folder_path}/model.keras", custom_objects={"loss": binary_focal_loss(gamma=2.0, alpha=0.8)})

with open(f"{model_folder_path}/history.json", 'r') as f:
    history = json.load(f)

## Plot the training metrics

In [ ]:


total_epochs = len(history['loss'])
hist_keys = list(history.keys())
hist_names = ['Loss', 'Accuracy', 'Precision', 'Recall', 'Binary accuracy']

for i in range(int(len(hist_keys)/2)):

    # Plotting the curves
    epochs_list = range(1, total_epochs + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs_list, history[hist_keys[i]], 'b-', label='Training ' + hist_names[i])
    plt.plot(epochs_list, history[hist_keys[i+int((len(hist_keys)/2))]], 'r-', label='Validation ' + hist_names[i])
    plt.title('Training and Validation '+ hist_names[i])
    plt.xlabel('Epochs')
    plt.ylabel(hist_names[i])
    plt.legend()
    plt.show()

## Comparison with the expected recipes

In [ ]:
comparison_df = pd.read_csv(f'{MODELS_PATH}/{MODEL_NAME}/comparison.csv')
comparison_df

In [ ]:
with open(f'{MODELS_PATH}/{MODEL_NAME}/comparison_summary.json', 'r') as f:
    comparison_summary = json.load(f) 

comparison_summary

## Confidence of the predictions

In [ ]:
confidence_df = pd.read_csv(f'{MODELS_PATH}/{MODEL_NAME}/confidence.csv')
confidence_df

Plot the confidence of True and False predictions

In [ ]:
# sort by mean_above with nan values handled
import numpy as np


stats_df = confidence_df#.sort_values(by='mean_above', ascending=False, na_position='last')

# plot again
fig, ax = plt.subplots(figsize=(10, 25))

y_pos = np.arange(len(stats_df))

# plot horizontal bars for below threshold
bars_below = ax.barh(
    y_pos - 0.2,
    stats_df['mean_below'],
    xerr=stats_df['std_below'],
    height=0.4,
    label='below threshold',
    align='center',
    error_kw=dict(ecolor='gray', lw=0.8)
)

# plot horizontal bars for above threshold
bars_above = ax.barh(
    y_pos + 0.2,
    stats_df['mean_above'],
    xerr=stats_df['std_above'],
    height=0.4,
    label='above threshold',
    align='center',
    error_kw=dict(ecolor='gray', lw=0.8)
)

# add vertical line for the threshold
ax.axvline(THRESHOLD, color='green', linestyle=':', linewidth=1.5, label=f'threshold = {THRESHOLD}')


# annotate bars with value ± std
for i, (bar_below, bar_above) in enumerate(zip(bars_below, bars_above)):
    mean_below = stats_df['mean_below'].iloc[i]
    std_below = stats_df['std_below'].iloc[i]
    mean_above = stats_df['mean_above'].iloc[i]
    std_above = stats_df['std_above'].iloc[i]

    if not np.isnan(mean_below):
        ax.text(bar_below.get_width() + 0.01, bar_below.get_y() + bar_below.get_height() / 2,
                f'{mean_below:.2f} ± {std_below:.2f}', va='center', fontsize=8)

    if not np.isnan(mean_above):
        ax.text(bar_above.get_width() + 0.01, bar_above.get_y() + bar_above.get_height() / 2,
                f'{mean_above:.2f} ± {std_above:.2f}', va='center', fontsize=8)

# set labels
ax.set_yticks(y_pos)
#ax.set_yticklabels(stats_df.index)
ax.set_yticklabels(y_test.columns)
ax.invert_yaxis()
ax.set_xlabel('Mean Value')
ax.set_title('Mean Values Above and Below Threshold with Std Dev')
ax.legend()

plt.tight_layout()
plt.show()

# exploiting and not exploring

# maybe drop the treshold and use the non tresholded prediction as probability to be present instead of the historic occurence

## Analyse the nb of occurence of each pigment

In [ ]:
y_bin_df = pd.read_csv(f'{base}/Dataset/traintest/y.csv')
y_bin_df[y_bin_df > 0] = 1
y_bin_df

In [ ]:
# sum each column
col_sums = y_bin_df.sum()
col_sums = col_sums[col_sums > 0].sort_values(ascending=True)

hidden_names = [f'pigment {i}' for i in range(len(col_sums), 0, -1)]

# plot with y-axis as column names and x-axis as sums
plt.figure(figsize=(7, len(col_sums) * 0.2))
plt.barh(hidden_names, col_sums.values)
plt.xlabel('present in recipes')
plt.ylabel('pigment')
plt.title(f'Amount of times a pigment is present in recipes\n{len(col_sums)} out of {y_bin_df.shape[1]} are present at least in one recipe')

# set x-axis limit with some padding for labels
plt.xlim(0, col_sums.max() * 1.25)

# add value labels after each bar
for i, v in enumerate(col_sums.values):
    plt.text(v + 0.05, i, f'{int(v)} ({v/len(y_bin_df)*100:.2f}%)', va='center')

plt.show()